<a href="https://colab.research.google.com/github/AyushiB56/Neural-Network/blob/main/vanilla_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import os
import glob
import string
import unicodedata
import random
import io

In [2]:

every_ascii_character= string.ascii_letters + " .,:;"
len_of_char= len(every_ascii_character)

In [3]:


def unicode_to_ascii(s):
  return ''. join(c for c in unicodedata.normalize('NFD',s) if  unicodedata.category(c) != 'Mn' and c in every_ascii_character)

In [4]:
def load_data():
  countries=[]
  country_people_name={}
  def find_files(path):
    return glob.glob(path)
  def read_lines(filename):
    lines= io.open(filename,encoding='utf-8').read().strip().split('\n')

    return [unicode_to_ascii(line) for line in lines]

  for files in find_files('sample_data/data/*.txt'):
    country =  os.path.splitext(os.path.basename(files))[0]

    countries.append(country)

    lines= read_lines(files)
    country_people_name[country]=lines
  return country_people_name,countries





In [5]:
import numpy as np

In [6]:
def find_letter(letter):
  return every_ascii_character.find(letter)

In [7]:
def letter_to_ohe(letter):
  letter_to_tensor= torch.zeros(1,len_of_char)
  letter_to_tensor[0][find_letter(letter)]=1
  return letter_to_tensor

In [8]:
def line_to_ohe(line):
  tensor = torch.zeros(len(line), 1, len_of_char)
  for i, letter in enumerate(line):
        tensor[i][0][find_letter(letter)] = 1
  return tensor




In [9]:
def random_set(country_people_name,countries):
  def random_choice(a):
    return a[random.randint(0,len(countries)-1)]
  country= random_choice(countries)
  country_tensor= torch.tensor([countries.index(country)],dtype=torch.long)

  people_name= random_choice(country_people_name[country])
  line_tensor= line_to_ohe(people_name)
  return country_tensor,country,people_name,line_tensor

In [10]:
def random_training_example(category_lines, all_categories):

    def random_choice(a):
        random_idx = random.randint(0, len(a) - 1)
        return a[random_idx]

    category = random_choice(all_categories)
    line = random_choice(category_lines[category])
    category_tensor = torch.tensor([all_categories.index(category)], dtype=torch.long)
    line_tensor = line_to_ohe(line)
    return category, line, category_tensor, line_tensor

In [11]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt




In [12]:
class Rnn_module(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):

    super(Rnn_module, self).__init__()
    self.hidden_size= hidden_size
    self.i2h=nn.Linear(input_size + hidden_size, hidden_size)
    self.i2o=nn.Linear(input_size + hidden_size, output_size)
    self.softmax= nn.LogSoftmax(dim=1)
  def forward(self, input_tensor, hidden_tensor):
    combined= torch.cat((input_tensor, hidden_tensor),1)
    hidden = torch.tanh(self.i2h(combined))
    output = self.i2o(combined)
    output = self.softmax(output)


    return output, hidden
  def init_hidden(self):
    return torch.zeros(1, self.hidden_size)


In [13]:
country_people_name,countries= load_data()


hidden_size=128
rnn= Rnn_module(len_of_char,hidden_size,len(countries))
optimizer= torch.optim.Adam(rnn.parameters(), lr=0.005)
loss= nn.NLLLoss()
def train(line_tensor,country_tensor):

  hidden= rnn.init_hidden()


  for i in range(len(line_tensor)):
      output, hidden = rnn(line_tensor[i], hidden)
  loss_value= loss(output,country_tensor)
  optimizer.zero_grad()
  loss_value.backward()
  optimizer.step()


    #print("loss per epoch", loss_value.item())
  return output,loss_value.item()



In [14]:
def category_output(output):
  max_out= torch.argmax(output).item()
  return countries[max_out]

In [15]:
loss_epoch= []

In [ ]:
all_loss=[]
current_loss=0
for i in range(100000):
  country_tensor,country,people_name,line_tensor= random_set(country_people_name,countries)
  output,train_loss=train(line_tensor,country_tensor)
  current_loss+=train_loss
  print(current_loss)

  if (i+1)%1000==0:
    all_loss.append(current_loss/1000)
    current_loss=0

Streaming output truncated to the last 5000 lines.
1038.8304837062751
1040.6807205989753
1041.3857286885177
1044.2894500210678
1044.3949371740018
1045.8484819099103
1049.9792792960798
1049.9953947044705
1051.2103297687863
1053.0032010055875
1053.4122628547047
1053.4863842703198
1057.0742138601636
1057.5260422207211
1062.4286582447385
1063.9156874395703
1064.103473139534
1064.1610268182849
1064.1691110130196
1066.9956057067757
1067.4016065593605
1067.4653112884407
1070.4482529159432
1073.5614769454842
1073.8054650570994
1077.6989773060923
1080.8189237858896
1086.4807184483652
1086.4868991450203
1089.4252815321815
1093.124709613694
1094.894333131684
1095.263486780299
1096.6774468795193
1096.8266800270212
1096.8399495930207
1098.8946149200929
1099.2193268985284
1099.225268129332
1099.8587645330263
1099.8727718862501
1103.7017969163862
1109.9604872735945
1114.1171778711287
1114.1289226292429
1116.8596193074045
1120.3067523716745
1120.4343242822943
1120.9978972255049
1125.8643532573042
1125

In [ ]:
plt.figure()
plt.plot(all_loss)
plt.show()

In [ ]:

def predict(input_line):
    print(f"\n> {input_line}")
    with torch.no_grad():
        line_tensor = line_to_ohe(input_line)

        hidden = rnn.init_hidden()

        for i in range(line_tensor.size()[0]):
            output, hidden = rnn(line_tensor[i], hidden)

        guess = category_output(output)
        print(guess)


In [ ]:
predict('Addis')